In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *
from sklearn.metrics import mean_absolute_error, mean_squared_error
from models import *
from plots import *

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
data = load_data()

In [ ]:
target = 't_seasdiff'
exog=['lagged_tmed', 'lagged_prec', 'lagged_tmin', 'lagged_tmax']

In [ ]:
data.head()

In [ ]:
if target == 't_seasdiff':
    data["t_seasdiff"]= data["tdiff"].diff(12)
    data = data.dropna(subset=["t_seasdiff"])

# SARIMA / SARIMAX

In [ ]:
p, d, q = 0, 1, 1
P, D, Q, s = 1, 0, 0, 12


In [ ]:
rmse, mae, results, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=(p,d,q),
    seasonal_order=(P,D,Q,12),
    rmse=rmse,
    mae=mae,
)

In [ ]:
rmse, mae, results, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s, exog= exog)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=(p,d,q),
    seasonal_order=(P,D,Q,12),
    rmse=rmse,
    mae=mae,
)

In [ ]:
from utils import sarima_candidates_from_acf

cand_res = sarima_candidates_from_acf(data[target], m=12, max_p=3, max_q=2, max_P=3, max_Q=3, max_lag=50)

candidates = cand_res["candidates"]
for i, (p, d, q, P, D, Q, s) in enumerate(candidates, 1):
    print(f"{i:02d}. SARIMA({p},{d},{q})({P},{D},{Q},{s})")

In [ ]:
best_info, scores_df = sarimax_grid_search(
    df=data,
    target=target,
    candidates=candidates,
    forecast_window=12,
    no_windows=10,
    exog=None,    # or list of exogenous column names
    metric="rmse",
    verbose=True,
)

In [ ]:
plot_sarimax_results(
    train=best_info["train"],
    test=best_info["test"],
    forecast=best_info["forecast"],
    results=best_info["results"],
    order=best_info["order"],
    seasonal_order=best_info["seasonal_order"],
    rmse=best_info["rmse"],
    mae=best_info["mae"],
)

In [ ]:
best_info, scores_df = sarimax_grid_search(
    df=data,
    target=target,
    candidates=candidates,
    forecast_window=12,
    no_windows=10,
    exog=None,    
    metric="rmse",
    verbose=True,
)

In [ ]:
plot_sarimax_results(
    train=train,
    test=best_info["test"],
    forecast=best_info["forecast"],
    results=best_info["results"],
    order=best_info["order"],
    seasonal_order=best_info["seasonal_order"],
    rmse=best_info["rmse"],
    mae=best_info["mae"],
)

In [ ]:
best_info, scores_df = sarimax_grid_search(
    df=data,
    target=target,
    candidates=candidates,
    forecast_window=12,
    no_windows=10,
    exog=exog,    # or list of exogenous column names
    metric="rmse",
    verbose=True,
)

In [ ]:
plot_sarimax_results(
    train=best_info["train"],
    test=best_info["test"],
    forecast=best_info["forecast"],
    results=best_info["results"],
    order=best_info["order"],
    seasonal_order=best_info["seasonal_order"],
    rmse=best_info["rmse"],
    mae=best_info["mae"],
)

# Using AutoSarima with the current train/test split

In [ ]:
rmse, mae, results, forecast, test, train, order, seasonal_order = auto_sarima_experiment(data, target, forecast_window=12, no_windows=10, m=12)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=order,
    seasonal_order=seasonal_order,
    rmse=rmse,
    mae=mae,
)

In [ ]:
rmse, mae, results, forecast, test, train, order, seasonal_order = auto_sarima_experiment(data, target, forecast_window=12, no_windows=10, m=12, exog=exog)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=order,
    seasonal_order=seasonal_order,
    rmse=rmse,
    mae=mae,
)

In [ ]:
h = 12
years_test = 2
years_train = 8
forecast_window = h * years_test
step = h
train_size = h * years_train
best_info, scores_df  = select_best_sarima_cv(
    data,
    target,
    candidates=candidates,
    forecast_window=forecast_window,
    train_size=train_size,
    step=step,
    exog=None,
)

In [ ]:
p, d, q = best_info["order"][0] , best_info["order"][1], best_info["order"][2]
P, D, Q, s = best_info["seasonal_order"][0], best_info["seasonal_order"][1], best_info["seasonal_order"][2], best_info["seasonal_order"][3]
rmse, mae, results_train, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results_train,     # ✅ use results_train
    order=(p, d, q),
    seasonal_order=(P, D, Q, s),
    rmse=rmse,
    mae=mae,
)

In [ ]:
best_info, scores_df  = select_best_sarima_cv(
    data,
    target,
    candidates=candidates,
    forecast_window=forecast_window,
    train_size=train_size,
    step=step,
    exog=exog,
)

In [ ]:
p, d, q = best_info["order"][0] , best_info["order"][1], best_info["order"][2]
P, D, Q, s = best_info["seasonal_order"][0], best_info["seasonal_order"][1], best_info["seasonal_order"][2], best_info["seasonal_order"][3]
rmse, mae, results_train, forecast, test, train = sarimax_experiment(data, target, p, d, q, P, D, Q, s)

In [ ]:
plot_sarimax_results(
    train=train,
    test=test,
    forecast=forecast,
    results=results,
    order=order,
    seasonal_order=seasonal_order,
    rmse=rmse,
    mae=mae,
)